# Data Loading and Geocoding

1. Load the data from the CSV
2. Format the data for geocoding
3. Call API
4. Add lat/lon to data
5. Load data into database

### Imports/Data Structures

Simple enough. Import the libraries and get some basics set up

In [ ]:
import csv
import sqlite3
import time

import requests
from requests.structures import CaseInsensitiveDict

API_KEY = "API_KEY"
base_url = "https://api.geoapify.com/v1/geocode/search?text="
headers = CaseInsensitiveDict()
headers["Accept"] = "application/json"
database_data = []

### Read Data and Call API

Use a csv reader to read the data into dictionaries. Next get the address from each line, and pass it into the API. Finally, put all of the row data into a tuple and append it to a list of all the rows

In [ ]:
with open('Location_Database.csv', newline='') as csvfile: # open file
    reader = csv.DictReader(csvfile) # csv reader. reads in whole file
    for row in reader: # iterate through the rows
        address = row["Address"] # get row address
        url = f"{base_url}{address}&apiKey={API_KEY}" # generate request url
        print(f"Getting location for: {address}")
        response = requests.get(url, headers) # API request
        if (response.status_code == 200):
            print(f"Found location for: {address}\n")
        else:
            print(f"Failed to find location for: {address}\n")

        data = response.json() # how do we interpret the data? JSON
        row["Longitude"] = data.get("features")[0].get("properties").get("lon") # extract Longitude from requested data
        row["Latitude"] = data.get("features")[0].get("properties").get("lat") # extract Latitude form requested data
        # format the data
        formatted_data = (row["Company Name"], row["Type"], row["Secondary Type"], row["Address"], int(row["zip code"]), float(row["Latitude"]), float(row["Longitude"]))
        database_data.append(formatted_data) # add formatted data to list
        time.sleep(1) # wait so the API doesn't get made and ratelimit

Getting location for: 5501 Midlothian Turnpike, Richmond, VA 23224
Found location for: 5501 Midlothian Turnpike, Richmond, VA 23224

Getting location for: 4819 Monument Ave, Richmond, VA 23226
Found location for: 4819 Monument Ave, Richmond, VA 23226

Getting location for: 1501 Decatur St, Richmond, VA 23224
Found location for: 1501 Decatur St, Richmond, VA 23224

Getting location for: 8020 Villa Park Dr, Richmond, VA 23228
Found location for: 8020 Villa Park Dr, Richmond, VA 23228

Getting location for: 1519 Williamsburg Rd, Richmond, VA 23231
Found location for: 1519 Williamsburg Rd, Richmond, VA 23231

Getting location for: 614 N 3rd St, Richmond, VA 23219
Found location for: 614 N 3rd St, Richmond, VA 23219

Getting location for: 3511 P St, Richmond, VA 23223
Found location for: 3511 P St, Richmond, VA 23223

Getting location for: 3613 Meadowbridge Rd, Richmond, VA 23222
Found location for: 3613 Meadowbridge Rd, Richmond, VA 23222

Getting location for: 8820 Metro Ct, Richmond, VA 

### Create the Database

Connect to a non-existent database. The file gets generated automatically if it doesn't exist. Then, create the table in the database and fill it with the data.

In [ ]:
con = sqlite3.connect("location_database.sqlite") # connect/create
cur = con.cursor() # how we interact with the database and run commands
cur.execute("CREATE TABLE locations(company_name, type, secondary_type, address, zip_code, latitude, longitude)") # create the table
cur.executemany("INSERT INTO locations VALUES(?, ?, ?, ?, ?, ?, ?)", database_data) # add the data. iterates through the list and extracts values from the row tuples
con.commit() # commit changes

In [ ]:
cur.execute("SELECT company_name, type, secondary_type, address FROM locations WHERE zip_code=23219") # run raw SQL
print(cur.fetchall()) # access query results

[('Third Street Bethel AME Church - Food Distribution Center', 'Food Bank', 'donation center', '614 N 3rd St, Richmond, VA 23219'), ('Commonwealth Catholic Charities - Housing Resource Center', 'Food Bank', 'donation center', '809 Oliver Hl Wy, Richmond, VA 23219')]


In [ ]:
zips = (23219,) # zip codes can be passed as a tuple
cur.execute("SELECT company_name, type, secondary_type, address FROM locations WHERE zip_code=?", zips) # allows for variables in queries
print(cur.fetchall())

[('Third Street Bethel AME Church - Food Distribution Center', 'Food Bank', 'donation center', '614 N 3rd St, Richmond, VA 23219'), ('Commonwealth Catholic Charities - Housing Resource Center', 'Food Bank', 'donation center', '809 Oliver Hl Wy, Richmond, VA 23219')]


In [ ]:
con.close() # gotta close the connection once you're done!